In [1]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("Hugging Face token configured successfully!")
else:
    print("Token not found. Check Colab Secrets.")

Hugging Face token configured successfully!


In [3]:
%pip install -q -U transformers datasets peft accelerate sentencepiece pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 17.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [4]:
import torch
import pandas as pd
import transformers
import datasets
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Pandas:", pd.__version__)
print("All required libraries imported successfully!")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


PyTorch: 2.11.0+cu130
GPU available: True
Pandas: 3.1.0
All required libraries imported successfully!


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Gemma-2B loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.


tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Gemma-2B loaded successfully!
GPU: Tesla T4


In [6]:
from datasets import load_dataset

dataset = load_dataset("rajpurkar/squad")

print(dataset)

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [7]:
print(dataset["train"][0])

{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


In [8]:
print("Training examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))

Training examples: 87599
Validation examples: 10570


In [9]:
train_data = dataset["train"].shuffle(
    seed=42
).select(range(500))

eval_data = dataset["validation"].shuffle(
    seed=42
).select(range(100))

print("Selected training examples:", len(train_data))
print("Selected validation examples:", len(eval_data))

Selected training examples: 500
Selected validation examples: 100


In [10]:
example = train_data[0]

print("CONTEXT:")
print(example["context"])

print("\nQUESTION:")
print(example["question"])

print("\nANSWER:")
print(example["answers"]["text"][0])

CONTEXT:
The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.

QUESTION:
What percentage of Egyptians polled support death penalty for those leaving Islam?

ANSWER:
84%


In [11]:
def format_example(example):
    answer = example["answers"]["text"][0]

    text = (
        "Answer the question using the context below.\n\n"
        f"Context: {example['context']}\n\n"
        f"Question: {example['question']}\n\n"
        f"Answer: {answer}"
    )

    return {"text": text}


train_data = train_data.map(format_example)
eval_data = eval_data.map(format_example)

print(train_data[0]["text"])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Answer the question using the context below.

Context: The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.

Question: What percentage of Egyptians polled support death penalty for those leaving Islam?

Answer: 84%


In [12]:
MAX_LENGTH = 384

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )


tokenized_train = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_eval = eval_data.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_data.column_names
)

print("Training dataset:", tokenized_train)
print("Validation dataset:", tokenized_eval)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Training dataset: Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 500
})
Validation dataset: Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 100
})


In [13]:
print("Token IDs:")
print(tokenized_train[0]["input_ids"][:20])

print("\nAttention mask:")
print(tokenized_train[0]["attention_mask"][:20])

print("\nTraining examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_eval))

Token IDs:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Attention mask:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Training examples: 500
Validation examples: 100


In [14]:
def generate_answer(context, question, max_new_tokens=40):
    prompt = (
        "Answer the question using the context below.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

In [15]:
context = (
    "The Eiffel Tower is located in Paris, France. "
    "It was completed in 1889."
)

question = "When was the Eiffel Tower completed?"

original_answer = generate_answer(context, question)

print("QUESTION:", question)
print("ORIGINAL MODEL ANSWER:", original_answer)

QUESTION: When was the Eiffel Tower completed?
ORIGINAL MODEL ANSWER: The Eiffel Tower was completed in 1889.

What is the difference between a declarative statement and an expression?

In the following sentence, underline the correct modifier from the pair given in


In [16]:
baseline_results = [{
    "context": context,
    "question": question,
    "original_answer": original_answer
}]

print(baseline_results)

[{'context': 'The Eiffel Tower is located in Paris, France. It was completed in 1889.', 'question': 'When was the Eiffel Tower completed?', 'original_answer': 'The Eiffel Tower was completed in 1889.\n\nWhat is the difference between a declarative statement and an expression?\n\nIn the following sentence, underline the correct modifier from the pair given in'}]


In [17]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA configuration created successfully!")

LoRA configuration created successfully!


In [19]:
%pip install -q -U torchao

model.config.use_cache = False
model.gradient_checkpointing_enable()

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.3 MB/s eta 0:00:00
trainable params: 921,600 || all params: 2,507,094,016 || trainable%: 0.0368


In [20]:
print("Training examples:", len(tokenized_train))
print("Evaluation examples:", len(tokenized_eval))

Training examples: 500
Evaluation examples: 100


In [21]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./gemma2b-lora-squad",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,
    optim="adamw_torch"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Training configuration ready!")

Training configuration ready!


In [22]:
print(tokenized_train.column_names)
print(tokenized_train[0].keys())
print(tokenizer.decode(tokenized_train[0]["input_ids"][:100]))

['input_ids', 'attention_mask']
dict_keys(['input_ids', 'attention_mask'])
<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


In [23]:
MAX_LENGTH = 384

def prepare_example(example):
    prompt = (
        "Answer the question using the context below.\n"
        f"Context: {example['context']}\n"
        f"Question: {example['question']}\n"
        "Answer:"
    )

    answer = example["answers"]["text"][0] if example["answers"]["text"] else ""

    prompt_tokens = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH - 64,
        add_special_tokens=True
    )

    answer_tokens = tokenizer(
        answer + tokenizer.eos_token,
        truncation=True,
        max_length=64,
        add_special_tokens=False
    )

    input_ids = prompt_tokens["input_ids"] + answer_tokens["input_ids"]
    labels = [-100] * len(prompt_tokens["input_ids"]) + answer_tokens["input_ids"]

    input_ids = input_ids[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]
    attention_mask = [1] * len(input_ids)

    padding_length = MAX_LENGTH - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * padding_length
    attention_mask += [0] * padding_length
    labels += [-100] * padding_length

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_data = dataset["train"].select(range(500))
eval_data = dataset["validation"].select(range(100))

tokenized_train = train_data.map(
    prepare_example,
    remove_columns=train_data.column_names
)

tokenized_eval = eval_data.map(
    prepare_example,
    remove_columns=eval_data.column_names
)

print(tokenized_train.column_names)
print("Training:", len(tokenized_train))
print("Evaluation:", len(tokenized_eval))

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'labels']
Training: 500
Evaluation: 100


In [24]:
from transformers import TrainingArguments, Trainer
import torch

training_args = TrainingArguments(
    output_dir="./gemma2b-lora-squad",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
    gradient_checkpointing=True,
    optim="adamw_torch",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval
)

print("Trainer created successfully!")

Trainer created successfully!


In [25]:
train_result = trainer.train()

print("Training completed!")
print(train_result.metrics)

Step,Training Loss
10,2.619055
20,1.810955
30,1.231032
40,0.827600
50,1.332215
60,1.147006
70,0.855297
80,0.754113
90,0.980962
100,0.870121


Training completed!
{'train_runtime': 182.5455, 'train_samples_per_second': 2.739, 'train_steps_per_second': 0.685, 'total_flos': 2284192530432000.0, 'train_loss': 1.1531323585510254, 'epoch': 1.0}


In [26]:
import os

SAVE_DIR = "./gemma2b-lora-squad/final_adapter"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Adapter saved successfully!")
print("Saved at:", SAVE_DIR)
print("Files:", os.listdir(SAVE_DIR))

Adapter saved successfully!
Saved at: ./gemma2b-lora-squad/final_adapter
Files: ['README.md', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [27]:
def test_finetuned_model(context, question):
    prompt = (
        "Answer the question using the context below.\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


context = (
    "The Eiffel Tower is located in Paris, France. "
    "It was completed in 1889 and is one of the most famous "
    "landmarks in the world."
)

question = "Where is the Eiffel Tower located?"

print("Question:", question)
print("Answer:", test_finetuned_model(context, question))

Question: Where is the Eiffel Tower located?
Answer: Paris, France


In [28]:
import re
import string
from collections import Counter

def normalize_answer(text):
    text = text.lower()
    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match(prediction, ground_truth):
    return int(
        normalize_answer(prediction)
        == normalize_answer(ground_truth)
    )


def token_f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    true_tokens = normalize_answer(ground_truth).split()

    common = Counter(pred_tokens) & Counter(true_tokens)
    overlap = sum(common.values())

    if not pred_tokens or not true_tokens:
        return float(pred_tokens == true_tokens)

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(true_tokens)

    return 2 * precision * recall / (precision + recall)


em_scores = []
f1_scores = []

for example in eval_data:
    context = example["context"]
    question = example["question"]
    answers = example["answers"]["text"]

    if not answers:
        continue

    prediction = test_finetuned_model(context, question)

    em = max(
        exact_match(prediction, answer)
        for answer in answers
    )
    f1 = max(
        token_f1(prediction, answer)
        for answer in answers
    )

    em_scores.append(em)
    f1_scores.append(f1)

print(f"Evaluation examples: {len(em_scores)}")
print(f"Exact Match (EM): {100 * sum(em_scores) / len(em_scores):.2f}%")
print(f"Token-level F1: {100 * sum(f1_scores) / len(f1_scores):.2f}%")

Evaluation examples: 100
Exact Match (EM): 91.00%
Token-level F1: 93.33%
